In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('Mall_Customers.csv')
print(df.head(10))
print('Shape:', df.shape)
print(df.dtypes)
print('Missing:', df.isnull().sum())
df = df.drop('CustomerID', axis=1)
df['Gender'] = df['Gender'].map({'Male':0,'Female':1})
print(df.describe())

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,4))
for ax,col in zip(axes,['Age','Annual Income (k$)','Spending Score (1-100)']):
    ax.hist(df[col],bins=20,color='steelblue',edgecolor='black'); ax.set_title(col)
plt.tight_layout(); plt.show()
plt.scatter(df['Annual Income (k$)'], df['Spending Score (1-100)']); plt.xlabel('Annual Income'); plt.ylabel('Spending Score'); plt.show()
df.boxplot(column='Spending Score (1-100)', by='Gender'); plt.show()

In [ ]:
features=['Gender','Age','Annual Income (k$)','Spending Score (1-100)']
X=StandardScaler().fit_transform(df[features])
wcss=[]; sil=[]
for k in range(2,11):
    km=KMeans(n_clusters=k,random_state=42,n_init=10); lbl=km.fit_predict(X)
    wcss.append(km.inertia_); sil.append(silhouette_score(X,lbl))
plt.plot(range(2,11),wcss,'bo-'); plt.title('Elbow'); plt.show()
plt.plot(range(2,11),sil,'go-'); plt.title('Silhouette'); plt.show()
kmeans=KMeans(n_clusters=5,random_state=42,n_init=10); df['Cluster']=kmeans.fit_predict(X)
centroids=kmeans.cluster_centers_
plt.scatter(df['Annual Income (k$)'],df['Spending Score (1-100)'],c=df['Cluster'],cmap='viridis')
inv=StandardScaler().fit(df[features]); c=inv.inverse_transform(centroids)
plt.scatter(c[:,2],c[:,3],c='red',marker='X',s=200); plt.show()
print(df.groupby('Cluster')[features].mean())
print('Final silhouette:', silhouette_score(X,df['Cluster']))